In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.font_manager import FontProperties
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False

# 定义基础路径
BASE_DIR = os.path.dirname(os.path.abspath(''))
if not BASE_DIR:
    BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "pipotsenvironmentallaw", "data", "air_quality")
RESULTS_DIR = os.path.join(BASE_DIR, "pipotsenvironmentallaw", "results", "imputed")
LEGAL_SUPERVISION_DIR = os.path.join(BASE_DIR, "pipotsenvironmentallaw", "results", "legal_supervision")
os.makedirs(LEGAL_SUPERVISION_DIR, exist_ok=True)

print("法律监督体系模块已加载")


In [ ]:
class EnvironmentalLegalStandards:
    """环境法律标准数据库"""
    
    def __init__(self):
        """初始化环境空气质量标准（GB 3095-2012）"""
        # 环境空气质量标准（GB 3095-2012）- 二级标准（适用于居住区、混合区、文化区等）
        self.standards = {
            'PM2.5': {
                '24h_mean': 75,  # 24小时平均浓度限值 (μg/m³)
                'annual_mean': 35,  # 年平均浓度限值 (μg/m³)
                'level_1': 35,  # 一级标准24h均值
                'level_2': 75,  # 二级标准24h均值
                'level_3': 115,  # 三级标准24h均值
                'legal_basis': 'GB 3095-2012《环境空气质量标准》',
                'penalty_basis': '《大气污染防治法》第九十九条'
            },
            'PM10': {
                '24h_mean': 150,  # 24小时平均浓度限值 (μg/m³)
                'annual_mean': 70,  # 年平均浓度限值 (μg/m³)
                'level_1': 70,  # 一级标准24h均值
                'level_2': 150,  # 二级标准24h均值
                'level_3': 250,  # 三级标准24h均值
                'legal_basis': 'GB 3095-2012《环境空气质量标准》',
                'penalty_basis': '《大气污染防治法》第九十九条'
            },
            'AQI': {
                'excellent': 50,  # 优
                'good': 100,  # 良
                'light_pollution': 150,  # 轻度污染
                'moderate_pollution': 200,  # 中度污染
                'heavy_pollution': 300,  # 重度污染
                'severe_pollution': 500,  # 严重污染
                'legal_basis': 'HJ 633-2012《环境空气质量指数（AQI）技术规定（试行）》',
                'penalty_basis': '《大气污染防治法》第九十九条'
            }
        }
        
        # 法律风险等级定义
        self.risk_levels = {
            'low': {'threshold': 0.8, 'description': '低风险：符合标准，无需采取行动'},
            'medium': {'threshold': 1.0, 'description': '中风险：接近标准限值，需要关注'},
            'high': {'threshold': 1.5, 'description': '高风险：超过标准限值，需要采取治理措施'},
            'critical': {'threshold': float('inf'), 'description': '严重风险：严重超标，可能面临法律处罚'}
        }
        
        # 处罚标准（参考《大气污染防治法》）
        self.penalty_standards = {
            'exceed_24h': {
                'description': '超过24小时平均浓度限值',
                'penalty_range': '10-100万元',
                'legal_basis': '《大气污染防治法》第九十九条'
            },
            'exceed_annual': {
                'description': '超过年平均浓度限值',
                'penalty_range': '50-500万元',
                'legal_basis': '《大气污染防治法》第九十九条'
            },
            'severe_pollution': {
                'description': '严重污染天气',
                'penalty_range': '100-1000万元',
                'legal_basis': '《大气污染防治法》第一百条'
            }
        }
    
    def get_standard(self, pollutant, standard_type='24h_mean'):
        """获取污染物标准限值"""
        if pollutant in self.standards:
            return self.standards[pollutant].get(standard_type, None)
        return None
    
    def get_risk_level(self, value, pollutant, standard_type='24h_mean'):
        """计算风险等级"""
        standard = self.get_standard(pollutant, standard_type)
        if standard is None:
            return None
        
        ratio = value / standard
        
        if ratio < self.risk_levels['low']['threshold']:
            return 'low'
        elif ratio < self.risk_levels['medium']['threshold']:
            return 'medium'
        elif ratio < self.risk_levels['high']['threshold']:
            return 'high'
        else:
            return 'critical'
    
    def get_risk_description(self, risk_level):
        """获取风险等级描述"""
        return self.risk_levels.get(risk_level, {}).get('description', '未知风险等级')

legal_standards = EnvironmentalLegalStandards()
print("环境法律标准数据库已初始化")


In [ ]:
class ComplianceChecker:
    """合规性检查器"""
    
    def __init__(self, legal_standards):
        self.standards = legal_standards
    
    def check_compliance(self, data_df, pollutant_col, date_col='date', station_col='station_id'):
        """检查数据合规性"""
        results = []
        
        for station in data_df[station_col].unique():
            station_data = data_df[data_df[station_col] == station].copy()
            
            for date in station_data[date_col].unique():
                daily_data = station_data[station_data[date_col] == date]
                
                # 计算24小时平均值
                daily_mean = daily_data[pollutant_col].mean()
                
                # 获取标准限值
                pollutant_name = pollutant_col
                standard = self.standards.get_standard(pollutant_name, '24h_mean')
                
                if standard is None:
                    continue
                
                # 计算超标倍数
                exceed_ratio = daily_mean / standard if standard > 0 else 0
                is_compliant = daily_mean <= standard
                
                # 获取风险等级
                risk_level = self.standards.get_risk_level(daily_mean, pollutant_name, '24h_mean')
                risk_desc = self.standards.get_risk_description(risk_level)
                
                results.append({
                    'station_id': station,
                    'date': date,
                    'pollutant': pollutant_name,
                    'daily_mean': daily_mean,
                    'standard_limit': standard,
                    'exceed_ratio': exceed_ratio,
                    'is_compliant': is_compliant,
                    'risk_level': risk_level,
                    'risk_description': risk_desc,
                    'max_value': daily_data[pollutant_col].max(),
                    'min_value': daily_data[pollutant_col].min(),
                    'data_points': len(daily_data)
                })
        
        return pd.DataFrame(results)
    
    def check_all_pollutants(self, data_df, pollutant_cols=['PM2.5', 'PM10', 'AQI']):
        """检查所有污染物的合规性"""
        all_results = []
        
        for pollutant in pollutant_cols:
            if pollutant in data_df.columns:
                results = self.check_compliance(data_df, pollutant)
                all_results.append(results)
        
        if all_results:
            return pd.concat(all_results, ignore_index=True)
        return pd.DataFrame()

compliance_checker = ComplianceChecker(legal_standards)
print("合规性检查器已初始化")


In [ ]:
class LegalRiskAssessor:
    """法律风险评估器"""
    
    def __init__(self, legal_standards):
        self.standards = legal_standards
    
    def assess_risk(self, compliance_results):
        """评估法律风险"""
        if compliance_results.empty:
            return pd.DataFrame()
        
        risk_assessments = []
        
        for _, row in compliance_results.iterrows():
            risk_score = self._calculate_risk_score(row)
            legal_consequences = self._assess_legal_consequences(row)
            recommended_actions = self._recommend_actions(row)
            
            risk_assessments.append({
                'station_id': row['station_id'],
                'date': row['date'],
                'pollutant': row['pollutant'],
                'risk_score': risk_score,
                'risk_level': row['risk_level'],
                'legal_consequences': legal_consequences,
                'recommended_actions': recommended_actions,
                'penalty_estimate': self._estimate_penalty(row),
                'compliance_status': '合规' if row['is_compliant'] else '超标'
            })
        
        return pd.DataFrame(risk_assessments)
    
    def _calculate_risk_score(self, row):
        """计算风险评分（0-100）"""
        if row['is_compliant']:
            base_score = 20  # 合规基础分
        else:
            base_score = 50 + min(50, row['exceed_ratio'] * 20)  # 超标基础分
        
        # 根据风险等级调整
        risk_multipliers = {
            'low': 0.5,
            'medium': 1.0,
            'high': 1.5,
            'critical': 2.0
        }
        
        multiplier = risk_multipliers.get(row['risk_level'], 1.0)
        return min(100, base_score * multiplier)
    
    def _assess_legal_consequences(self, row):
        """评估法律后果"""
        if row['is_compliant']:
            return "无法律风险，符合环境空气质量标准"
        
        consequences = []
        
        if row['exceed_ratio'] >= 1.5:
            consequences.append("可能面临行政处罚（10-100万元）")
            consequences.append("可能被责令限期治理")
        
        if row['exceed_ratio'] >= 2.0:
            consequences.append("可能面临更严厉的行政处罚（50-500万元）")
            consequences.append("可能被责令停产整治")
        
        if row['pollutant'] == 'AQI' and row['daily_mean'] >= 300:
            consequences.append("严重污染天气，可能启动应急响应")
            consequences.append("可能面临最高1000万元罚款")
        
        return "; ".join(consequences) if consequences else "轻微超标，建议加强监测"
    
    def _recommend_actions(self, row):
        """推荐应对措施"""
        actions = []
        
        if not row['is_compliant']:
            actions.append("立即采取减排措施")
            actions.append("加强监测频率")
            
            if row['risk_level'] == 'high' or row['risk_level'] == 'critical':
                actions.append("启动应急预案")
                actions.append("向环保部门报告")
                actions.append("考虑临时停产或限产")
        
        if row['risk_level'] == 'medium':
            actions.append("加强日常监管")
            actions.append("预防性减排措施")
        
        return "; ".join(actions) if actions else "保持现状，继续监测"
    
    def _estimate_penalty(self, row):
        """估算可能的处罚金额"""
        if row['is_compliant']:
            return "无处罚"
        
        exceed_ratio = row['exceed_ratio']
        
        if exceed_ratio < 1.2:
            return "10-50万元"
        elif exceed_ratio < 1.5:
            return "50-100万元"
        elif exceed_ratio < 2.0:
            return "100-300万元"
        else:
            return "300-1000万元"

risk_assessor = LegalRiskAssessor(legal_standards)
print("法律风险评估器已初始化")


In [ ]:
class LegalSupervisionVisualizer:
    """法律监督可视化器"""
    
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def plot_compliance_status(self, compliance_results, title="合规性检查结果"):
        """绘制合规性状态图"""
        if compliance_results.empty:
            print("无数据可可视化")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. 合规性统计
        compliance_counts = compliance_results['is_compliant'].value_counts()
        axes[0, 0].pie(compliance_counts.values, labels=['合规', '超标'], 
                       autopct='%1.1f%%', startangle=90)
        axes[0, 0].set_title('合规性总体统计')
        
        # 2. 风险等级分布
        risk_counts = compliance_results['risk_level'].value_counts()
        risk_order = ['low', 'medium', 'high', 'critical']
        risk_labels = ['低风险', '中风险', '高风险', '严重风险']
        risk_data = [risk_counts.get(level, 0) for level in risk_order]
        axes[0, 1].bar(risk_labels, risk_data, color=['green', 'yellow', 'orange', 'red'])
        axes[0, 1].set_title('风险等级分布')
        axes[0, 1].set_ylabel('数量')
        
        # 3. 超标倍数分布
        exceed_data = compliance_results[~compliance_results['is_compliant']]['exceed_ratio']
        if len(exceed_data) > 0:
            axes[1, 0].hist(exceed_data, bins=20, color='red', alpha=0.7)
            axes[1, 0].axvline(x=1.0, color='black', linestyle='--', label='标准限值')
            axes[1, 0].set_title('超标倍数分布')
            axes[1, 0].set_xlabel('超标倍数')
            axes[1, 0].set_ylabel('频次')
            axes[1, 0].legend()
        
        # 4. 污染物超标情况
        pollutant_exceed = compliance_results.groupby('pollutant')['is_compliant'].apply(
            lambda x: (~x).sum()
        )
        axes[1, 1].bar(pollutant_exceed.index, pollutant_exceed.values, color='coral')
        axes[1, 1].set_title('各污染物超标次数')
        axes[1, 1].set_ylabel('超标次数')
        axes[1, 1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, f'{title}_compliance_status.png'), dpi=300)
        plt.close()
        print(f"合规性状态图已保存: {title}_compliance_status.png")
    
    def plot_risk_timeline(self, compliance_results, station_id=None):
        """绘制风险时间线"""
        if compliance_results.empty:
            return
        
        data = compliance_results.copy()
        if station_id:
            data = data[data['station_id'] == station_id]
        
        if data.empty:
            return
        
        # 转换日期
        data['date'] = pd.to_datetime(data['date'], format='%Y%m%d', errors='coerce')
        data = data.sort_values('date')
        
        fig, axes = plt.subplots(len(data['pollutant'].unique()), 1, 
                                figsize=(16, 6 * len(data['pollutant'].unique())))
        
        if len(data['pollutant'].unique()) == 1:
            axes = [axes]
        
        for idx, pollutant in enumerate(data['pollutant'].unique()):
            pollutant_data = data[data['pollutant'] == pollutant]
            
            # 风险等级颜色映射
            color_map = {
                'low': 'green',
                'medium': 'yellow',
                'high': 'orange',
                'critical': 'red'
            }
            
            # 绘制每日平均值
            axes[idx].plot(pollutant_data['date'], pollutant_data['daily_mean'], 
                          'o-', label='日均值', alpha=0.7)
            
            # 绘制标准限值线
            standard = legal_standards.get_standard(pollutant, '24h_mean')
            if standard:
                axes[idx].axhline(y=standard, color='red', linestyle='--', 
                                label=f'标准限值 ({standard})')
            
            # 根据风险等级着色
            for _, row in pollutant_data.iterrows():
                color = color_map.get(row['risk_level'], 'gray')
                axes[idx].scatter(row['date'], row['daily_mean'], 
                                color=color, s=100, alpha=0.6, 
                                edgecolors='black', linewidths=1)
            
            axes[idx].set_title(f'{pollutant} 风险时间线 - {station_id if station_id else "所有站点"}')
            axes[idx].set_xlabel('日期')
            axes[idx].set_ylabel(f'{pollutant} 浓度 (μg/m³)')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)
            axes[idx].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        filename = f'risk_timeline_{station_id if station_id else "all"}.png'
        plt.savefig(os.path.join(self.output_dir, filename), dpi=300)
        plt.close()
        print(f"风险时间线图已保存: {filename}")
    
    def plot_station_comparison(self, compliance_results):
        """绘制站点对比图"""
        if compliance_results.empty:
            return
        
        # 计算每个站点的超标率
        station_stats = compliance_results.groupby(['station_id', 'pollutant']).agg({
            'is_compliant': lambda x: (~x).sum() / len(x) * 100  # 超标率
        }).reset_index()
        station_stats.columns = ['station_id', 'pollutant', 'exceed_rate']
        
        # 选择超标率最高的前10个站点
        top_stations = station_stats.nlargest(10, 'exceed_rate')
        
        fig, ax = plt.subplots(figsize=(14, 8))
        
        stations = top_stations['station_id'].unique()
        pollutants = top_stations['pollutant'].unique()
        
        x = np.arange(len(stations))
        width = 0.35
        
        for i, pollutant in enumerate(pollutants):
            pollutant_data = top_stations[top_stations['pollutant'] == pollutant]
            values = [pollutant_data[pollutant_data['station_id'] == s]['exceed_rate'].values[0] 
                     if len(pollutant_data[pollutant_data['station_id'] == s]) > 0 else 0 
                     for s in stations]
            ax.bar(x + i * width, values, width, label=pollutant)
        
        ax.set_xlabel('监测站点')
        ax.set_ylabel('超标率 (%)')
        ax.set_title('各站点超标率对比（前10名）')
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels(stations, rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, 'station_comparison.png'), dpi=300)
        plt.close()
        print("站点对比图已保存: station_comparison.png")

visualizer = LegalSupervisionVisualizer(LEGAL_SUPERVISION_DIR)
print("法律监督可视化器已初始化")


In [ ]:
class LegalSupervisionReportGenerator:
    """法律监督报告生成器"""
    
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def generate_report(self, compliance_results, risk_assessments, 
                       start_date=None, end_date=None):
        """生成法律监督报告"""
        report_lines = []
        
        # 报告头部
        report_lines.append("=" * 80)
        report_lines.append("环境空气质量法律监督报告")
        report_lines.append("=" * 80)
        report_lines.append(f"生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        if start_date and end_date:
            report_lines.append(f"监测期间: {start_date} 至 {end_date}")
        report_lines.append("")
        
        # 执行摘要
        report_lines.append("一、执行摘要")
        report_lines.append("-" * 80)
        total_records = len(compliance_results)
        compliant_records = compliance_results['is_compliant'].sum()
        exceed_records = total_records - compliant_records
        compliance_rate = (compliant_records / total_records * 100) if total_records > 0 else 0
        
        report_lines.append(f"总监测记录数: {total_records}")
        report_lines.append(f"合规记录数: {compliant_records} ({compliance_rate:.2f}%)")
        report_lines.append(f"超标记录数: {exceed_records} ({100-compliance_rate:.2f}%)")
        report_lines.append("")
        
        # 风险等级统计
        report_lines.append("二、风险等级统计")
        report_lines.append("-" * 80)
        risk_counts = compliance_results['risk_level'].value_counts()
        risk_labels = {'low': '低风险', 'medium': '中风险', 'high': '高风险', 'critical': '严重风险'}
        for level, count in risk_counts.items():
            label = risk_labels.get(level, level)
            percentage = (count / total_records * 100) if total_records > 0 else 0
            report_lines.append(f"{label}: {count} 次 ({percentage:.2f}%)")
        report_lines.append("")
        
        # 超标情况分析
        report_lines.append("三、超标情况分析")
        report_lines.append("-" * 80)
        exceed_data = compliance_results[~compliance_results['is_compliant']]
        if len(exceed_data) > 0:
            report_lines.append(f"最大超标倍数: {exceed_data['exceed_ratio'].max():.2f}")
            report_lines.append(f"平均超标倍数: {exceed_data['exceed_ratio'].mean():.2f}")
            report_lines.append("")
            
            # 按污染物统计
            report_lines.append("各污染物超标情况:")
            for pollutant in exceed_data['pollutant'].unique():
                pollutant_exceed = exceed_data[exceed_data['pollutant'] == pollutant]
                report_lines.append(f"  {pollutant}: {len(pollutant_exceed)} 次超标")
                report_lines.append(f"    平均超标倍数: {pollutant_exceed['exceed_ratio'].mean():.2f}")
        else:
            report_lines.append("监测期间未发现超标情况")
        report_lines.append("")
        
        # 法律风险评估
        if not risk_assessments.empty:
            report_lines.append("四、法律风险评估")
            report_lines.append("-" * 80)
            critical_risks = risk_assessments[risk_assessments['risk_level'].isin(['high', 'critical'])]
            if len(critical_risks) > 0:
                report_lines.append(f"高风险/严重风险事件: {len(critical_risks)} 起")
                report_lines.append("")
                report_lines.append("高风险事件详情:")
                for idx, (_, row) in enumerate(critical_risks.head(10).iterrows(), 1):
                    report_lines.append(f"{idx}. {row['station_id']} - {row['date']} - {row['pollutant']}")
                    report_lines.append(f"   风险评分: {row['risk_score']:.1f}")
                    report_lines.append(f"   法律后果: {row['legal_consequences']}")
                    report_lines.append(f"   建议措施: {row['recommended_actions']}")
                    report_lines.append(f"   预估处罚: {row['penalty_estimate']}")
                    report_lines.append("")
            else:
                report_lines.append("未发现高风险事件")
            report_lines.append("")
        
        # 建议措施
        report_lines.append("五、建议措施")
        report_lines.append("-" * 80)
        if exceed_records > 0:
            report_lines.append("1. 对超标站点加强监测频率")
            report_lines.append("2. 对高风险站点采取紧急减排措施")
            report_lines.append("3. 建立超标预警机制")
            report_lines.append("4. 定期向环保部门报告监测数据")
            report_lines.append("5. 制定并实施污染治理方案")
        else:
            report_lines.append("监测期间数据合规，建议继续保持并加强日常监管")
        report_lines.append("")
        
        # 法律依据
        report_lines.append("六、法律依据")
        report_lines.append("-" * 80)
        report_lines.append("1. GB 3095-2012《环境空气质量标准》")
        report_lines.append("2. HJ 633-2012《环境空气质量指数（AQI）技术规定（试行）》")
        report_lines.append("3. 《中华人民共和国大气污染防治法》")
        report_lines.append("4. 《中华人民共和国环境保护法》")
        report_lines.append("")
        
        report_lines.append("=" * 80)
        
        # 保存报告
        report_text = "\n".join(report_lines)
        report_filename = f"legal_supervision_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        report_path = os.path.join(self.output_dir, report_filename)
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        
        print(f"法律监督报告已保存: {report_path}")
        return report_path

report_generator = LegalSupervisionReportGenerator(LEGAL_SUPERVISION_DIR)
print("法律监督报告生成器已初始化")


In [ ]:
def run_legal_supervision(data_dir=None, output_dir=None):
    """运行完整的法律监督流程"""
    if data_dir is None:
        data_dir = DATA_DIR
    if output_dir is None:
        output_dir = LEGAL_SUPERVISION_DIR
    
    print("=" * 80)
    print("开始执行法律监督检查...")
    print("=" * 80)
    
    # 1. 加载数据
    print("\n步骤1: 加载空气质量数据...")
    particulate_files = [f for f in os.listdir(data_dir) 
                        if f.endswith('_particulate.csv')]
    
    if not particulate_files:
        print("错误: 未找到数据文件！")
        return None
    
    print(f"找到 {len(particulate_files)} 个数据文件")
    
    # 合并所有数据
    all_data = []
    for file_name in particulate_files:
        file_path = os.path.join(data_dir, file_name)
        try:
            df = pd.read_csv(file_path)
            all_data.append(df)
        except Exception as e:
            print(f"警告: 无法读取文件 {file_name}: {str(e)}")
    
    if not all_data:
        print("错误: 无法加载任何数据！")
        return None
    
    combined_data = pd.concat(all_data, ignore_index=True)
    print(f"成功加载数据: {len(combined_data)} 条记录")
    
    # 2. 合规性检查
    print("\n步骤2: 执行合规性检查...")
    compliance_results = compliance_checker.check_all_pollutants(
        combined_data, 
        pollutant_cols=['PM2.5', 'PM10', 'AQI']
    )
    
    if compliance_results.empty:
        print("警告: 合规性检查未产生结果")
        return None
    
    print(f"完成合规性检查: {len(compliance_results)} 条检查结果")
    
    # 保存合规性检查结果
    compliance_file = os.path.join(output_dir, 'compliance_results.csv')
    compliance_results.to_csv(compliance_file, index=False, encoding='utf-8-sig')
    print(f"合规性检查结果已保存: {compliance_file}")
    
    # 3. 法律风险评估
    print("\n步骤3: 执行法律风险评估...")
    risk_assessments = risk_assessor.assess_risk(compliance_results)
    
    if not risk_assessments.empty:
        risk_file = os.path.join(output_dir, 'risk_assessments.csv')
        risk_assessments.to_csv(risk_file, index=False, encoding='utf-8-sig')
        print(f"风险评估结果已保存: {risk_file}")
    
    # 4. 生成可视化
    print("\n步骤4: 生成可视化图表...")
    visualizer.plot_compliance_status(compliance_results, "法律监督合规性检查")
    visualizer.plot_station_comparison(compliance_results)
    
    # 为每个站点生成时间线（选择前5个站点）
    top_stations = compliance_results['station_id'].value_counts().head(5).index
    for station in top_stations:
        visualizer.plot_risk_timeline(compliance_results, station)
    
    # 5. 生成报告
    print("\n步骤5: 生成法律监督报告...")
    start_date = combined_data['date'].min() if 'date' in combined_data.columns else None
    end_date = combined_data['date'].max() if 'date' in combined_data.columns else None
    
    report_path = report_generator.generate_report(
        compliance_results, 
        risk_assessments,
        start_date=start_date,
        end_date=end_date
    )
    
    print("\n" + "=" * 80)
    print("法律监督检查完成！")
    print("=" * 80)
    print(f"\n结果文件保存在: {output_dir}")
    print(f"- 合规性检查结果: compliance_results.csv")
    if not risk_assessments.empty:
        print(f"- 风险评估结果: risk_assessments.csv")
    print(f"- 法律监督报告: {os.path.basename(report_path)}")
    print(f"- 可视化图表: 保存在 {output_dir}")
    
    return {
        'compliance_results': compliance_results,
        'risk_assessments': risk_assessments,
        'report_path': report_path
    }

print("法律监督主函数已定义")


In [ ]:
# 执行法律监督检查
if __name__ == "__main__":
    results = run_legal_supervision()
    
    if results:
        print("\n法律监督检查摘要:")
        print(f"- 合规性检查记录数: {len(results['compliance_results'])}")
        if not results['risk_assessments'].empty:
            print(f"- 风险评估记录数: {len(results['risk_assessments'])}")
            critical_risks = results['risk_assessments'][
                results['risk_assessments']['risk_level'].isin(['high', 'critical'])
            ]
            print(f"- 高风险/严重风险事件: {len(critical_risks)} 起")
